# 🚀 JalanCerdas AI — Retrain Model v2

**YOLOv8s (small)** — lebih akurat dari YOLOv8n yang sekarang

### Cara Pakai:
1. **Set Runtime ke GPU**: Runtime → Change runtime type → **T4 GPU**
2. **Run semua cell** dari atas ke bawah (Shift+Enter)
3. **Download model** yang dihasilkan di cell terakhir
4. **Copy ke project** lalu rebuild APK

⏱️ **Estimasi waktu**: 30-50 menit (GPU T4)

## 1️⃣ Install Dependencies

In [ ]:
# Install all dependencies
!pip install -q ultralytics kagglehub pyyaml

# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")
else:
    print("❌ NO GPU! Go to: Runtime → Change runtime type → T4 GPU")
    print("   Then run this cell again.")

## 2️⃣ Download Dataset dari Kaggle

In [ ]:
import kagglehub

print("📥 Downloading dataset...")
path = kagglehub.dataset_download("habibiahmadim09/kerusakan-jalan")
print(f"✅ Dataset downloaded: {path}")

# Check dataset structure
import os
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(path, split)
    if os.path.exists(split_path):
        imgs = len([f for f in os.listdir(split_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
        xmls = len([f for f in os.listdir(split_path) if f.endswith('.xml')])
        print(f"  {split}: {imgs} images, {xmls} annotations")

## 3️⃣ Clone Repo & Prepare Dataset

In [ ]:
# Clone repo for config files
!git clone https://github.com/Srjeff27/jalancerdas-ai.git

# Convert VOC XML → YOLO format
import xml.etree.ElementTree as ET
import shutil
from pathlib import Path

CLASS_NAMES = [
    "retak_memanjang",
    "pengelupasan_lapisan_permukaan",
    "lubang",
    "retak_kulit_buaya",
    "retak_blok",
    "retak_pinggir",
]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

source = Path(path)
dataset_dir = Path("jalancerdas-ai/ai-model/dataset")

# Clean and create
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)

split_map = {"train": "train", "valid": "val", "test": "test"}
total_imgs, total_lbls = 0, 0

for src_split, dst_split in split_map.items():
    src_dir = source / src_split
    dst_img = dataset_dir / dst_split / "images"
    dst_lbl = dataset_dir / dst_split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    n_img, n_lbl = 0, 0
    for xml_file in sorted(src_dir.glob("*.xml")):
        # Find image
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = src_dir / (xml_file.stem + ext)
            if candidate.exists():
                img_path = candidate
                break
        if img_path is None:
            continue

        # Convert XML → YOLO
        tree = ET.parse(xml_file)
        root = tree.getroot()
        w = int(root.find("size/width").text)
        h = int(root.find("size/height").text)

        lines = []
        for obj in root.findall("object"):
            name = obj.find("name").text
            if name not in CLASS_TO_ID:
                continue
            cls_id = CLASS_TO_ID[name]
            bbox = obj.find("bndbox")
            xmin, ymin = float(bbox.find("xmin").text), float(bbox.find("ymin").text)
            xmax, ymax = float(bbox.find("xmax").text), float(bbox.find("ymax").text)
            cx, cy = ((xmin+xmax)/2)/w, ((ymin+ymax)/2)/h
            bw, bh = (xmax-xmin)/w, (ymax-ymin)/h
            lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        shutil.copy2(img_path, dst_img / img_path.name)
        (dst_lbl / (xml_file.stem + ".txt")).write_text("\n".join(lines))
        n_img += 1
        n_lbl += len(lines)

    total_imgs += n_img
    total_lbls += n_lbl
    print(f"  {dst_split}: {n_img} images, {n_lbl} labels")

print(f"\n✅ Dataset ready: {total_imgs} images, {total_lbls} labels")

## 4️⃣ Train YOLOv8s (200 epochs)

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s (small) — better accuracy than nano
model = YOLO("yolov8s.pt")

print("🚀 Training YOLOv8s...")
print("   This will take 30-50 minutes on GPU T4")
print("   Do NOT close this tab!\n")

results = model.train(
    data="jalancerdas-ai/ai-model/configs/data.yaml",
    epochs=200,
    batch=16,
    imgsz=640,
    device=0,
    patience=50,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    copy_paste=0.1,
    close_mosaic=15,
    project="jalancerdas-ai/ai-model/runs",
    name="train_v2",
    exist_ok=True,
)

best_pt = "jalancerdas-ai/ai-model/runs/train_v2/weights/best.pt"
print(f"\n✅ Training done! Best model: {best_pt}")

## 5️⃣ Evaluate Model

In [ ]:
model = YOLO(best_pt)
results = model.val(data="jalancerdas-ai/ai-model/configs/data.yaml", device=0)

print("\n📊 EVALUATION RESULTS")
print("=" * 50)
print(f"  mAP@50:     {results.box.map50:.4f}")
print(f"  mAP@50-95:  {results.box.map:.4f}")
print(f"  Precision:  {results.box.mp:.4f}")
print(f"  Recall:     {results.box.mr:.4f}")

f1 = 2 * (results.box.mp * results.box.mr) / (results.box.mp + results.box.mr + 1e-8)
print(f"  F1 Score:   {f1:.4f}")

print("\n  Per-class mAP@50:")
for i, ap in enumerate(results.box.ap50):
    print(f"    {results.names[i]:<40} {ap:.4f}")

print("\n" + "=" * 50)

## 6️⃣ Export to TFLite

In [ ]:
model = YOLO(best_pt)
export_path = model.export(
    format="tflite",
    imgsz=640,
    half=True,
    simplify=True,
)

import os
size_mb = os.path.getsize(export_path) / (1024*1024)
print(f"\n✅ TFLite exported!")
print(f"   Path: {export_path}")
print(f"   Size: {size_mb:.2f} MB")

## 7️⃣ Download Model Files

Click the download links below, then copy to your project.

In [ ]:
from google.colab import files

print("📥 Downloading model files...")
print("\nAfter download, copy to:")
print(f"  {export_path}  →  mobile/assets/models/pothole_yolo.tflite")
print(f"  {best_pt}  →  ai-model/best_retrained.pt\n")

# Download TFLite
files.download(export_path)

# Download best.pt
files.download(best_pt)

print("\n✅ Done! Now copy files to your project and rebuild APK.")

## 📋 Next Steps

After downloading the model:

```bash
# 1. Copy TFLite to mobile assets
cp best.tflite ~/Project/jalancerdas-ai/mobile/assets/models/pothole_yolo.tflite

# 2. Rebuild APK
cd ~/Project/jalancerdas-ai/mobile
flutter clean && flutter pub get && flutter build apk --release

# 3. Test on phone!
```